# 05 -- Tensor Decomposition

Build the multi-dialect tensor T from transformation matrices, perform Tucker
and CP decompositions, and analyse the resulting factors.

**Key modules:** `tensor.construction`, `tensor.tucker`, `tensor.cp`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from eigendialectos.constants import DialectCode, DIALECT_NAMES
from eigendialectos.types import EmbeddingMatrix, TransformationMatrix
from eigendialectos.spectral.transformation import compute_all_transforms
from eigendialectos.tensor.construction import build_dialect_tensor, extract_slice

## Build the Dialectal Tensor

T in R^{d x d x m} where m = 8 dialect varieties.

In [ ]:
rng = np.random.default_rng(42)
dim, vocab_size = 15, 60
vocab = [f"w{i}" for i in range(vocab_size)]

base = rng.standard_normal((dim, vocab_size)).astype(np.float64)
embeddings = {}
for code in DialectCode:
    noise = np.zeros_like(base) if code == DialectCode.ES_PEN else (
        rng.standard_normal((dim, vocab_size)) * 0.15
    )
    embeddings[code] = EmbeddingMatrix(data=base + noise, vocab=vocab, dialect_code=code)

transforms = compute_all_transforms(embeddings, reference=DialectCode.ES_PEN)
tensor = build_dialect_tensor(transforms)

print(f"Tensor shape: {tensor.shape}")
print(f"Dialect ordering: {[c.value for c in tensor.dialect_codes]}")

## Tensor Slice Norms

In [ ]:
for i, code in enumerate(tensor.dialect_codes):
    slice_norm = np.linalg.norm(tensor.data[:, :, i], "fro")
    dist_from_I = np.linalg.norm(tensor.data[:, :, i] - np.eye(dim), "fro")
    print(f"  {code.value}: ||W||_F = {slice_norm:.4f}, ||W-I||_F = {dist_from_I:.4f}")

## Tucker Decomposition

T ~ G x_1 A x_2 B x_3 C

In [ ]:
from eigendialectos.tensor.tucker import tucker_decompose, explained_variance

tucker_result = tucker_decompose(tensor, ranks=(10, 10, 4))
core = tucker_result["core_tensor"]
factors = tucker_result["factor_matrices"]

print(f"Core tensor shape: {core.shape}")
print(f"Factor matrix shapes: {[f.shape for f in factors]}")
print(f"Reconstruction error: {tucker_result['reconstruction_error']:.6f}")

ev = explained_variance(tensor, core, factors)
print(f"Explained variance: {ev:.4f} ({ev*100:.1f}%)")

## Mode-3 Factor Analysis

The mode-3 factor C (m x r3) reveals dialect groupings in reduced space.

In [ ]:
C = factors[2]  # shape: (m, r3)
print(f"Mode-3 factor shape: {C.shape}")

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(C, cmap="RdBu_r", aspect="auto")
ax.set_yticks(range(len(tensor.dialect_codes)))
ax.set_yticklabels([c.value for c in tensor.dialect_codes])
ax.set_xlabel("Factor index")
ax.set_title("Mode-3 Factor Loadings (Dialect Groupings)")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## CP Decomposition

T ~ sum_r w_r * (a_r x b_r x c_r)

In [ ]:
from eigendialectos.tensor.cp import cp_decompose

cp_result = cp_decompose(tensor, rank=5, n_restarts=3)
weights = cp_result["weights"]
cp_factors = cp_result["factor_matrices"]

print(f"CP weights: {weights.round(4)}")
print(f"CP factor shapes: {[f.shape for f in cp_factors]}")
print(f"Reconstruction error: {cp_result['reconstruction_error']:.6f}")

## Reconstruction Error vs Rank

In [ ]:
ranks_to_test = [1, 2, 3, 5, 8, 10]
errors = []
for r in ranks_to_test:
    try:
        res = cp_decompose(tensor, rank=r, n_restarts=2)
        errors.append(res["reconstruction_error"])
    except Exception:
        errors.append(np.nan)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ranks_to_test, errors, "o-", color="crimson")
ax.set_xlabel("CP Rank")
ax.set_ylabel("Reconstruction Error (Frobenius)")
ax.set_title("CP Decomposition: Rank vs Error")
plt.tight_layout()
plt.show()